# Papal visits: does the Vatican pursue a geopolitical objective?

Following Pope Leo's visit to France in Fall 2026 and given the media coverage it received, we decided to investigate the Vatican's diplomacy and to determine whether it pursues a geopolitical goal. To do this, we first analysed the countries visited by John-Paul II, Benedcit XVI and Francis. We cross the list of countries visited by these popes with the religious composition of these countries and their level of development.

Questions:

1. **Who do the popes speak to?** Are the visited countries mostly Catholic, or at least more Catholic than the world average?
2. **Growth or decline?** Do the popes go where Catholicism is growing or where it is declining?
3. **Development**: do they rather go to rich countries or to poor countries?

To do: the other religions and geography.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import s3fs

fs = s3fs.S3FileSystem(client_kwargs={"endpoint_url": "https://minio.lab.sspcloud.fr"})

POPES = ["John-Paul II", "Benedict XVI", "Francis"]
# Start and end of each pontificate
PONTIFICATES = {"John-Paul II": (1978, 2005), "Benedict XVI": (2005, 2013), "Francis": (2013, 2025)}

## 1. The data

| Table | Source | Content | Years |
| --- | --- | --- | --- |
| Trips | vatican.va (scripts in `acquisition/`) | one row per (trip, country) | 1979-2024 |
| Religions and population | Religious Characteristics of States Dataset, Demographics v. 2.0 (RCS-Dem 2.0), SPSS file | population and share of each religion, **including Catholics** | every year, up to 2015 |
| GDP, region, income | World Bank | | 1960-2024 |

### The trips

In [ ]:
with fs.open("avibert/diffusion/df_trips_countries.parquet", "rb") as f:
    trips = pd.read_parquet(f)

trips = trips.rename(columns={"Pope": "pope", "Year": "year", "Place": "trip",
                              "Country": "country", "ISO3": "iso3"})
trips["year"] = trips["year"].astype(int)
trips.head()

### The religions: RCS-Dem 2.0

One row per country and per year. The columns ending in `PC` are shares of the population (in %), `TOTPP` is the total population. We keep the main religions and the detail of Christians. The rest of the population is put in an `other_religions` column (Chinese folk religions, African religions, syncretic Christians...), except the people whose religion is unknown (`UNKPC`). Be careful: `CHRPC` (all Christians) already contains `CATPC` (Catholics). To find the majority religion, we must only compare the main religions, otherwise we mix levels.

In [ ]:
FILE_RCS = ("avibert/diffusion/Religious Characteristics of States Dataset Project - "
            "Demographics v. 2.0 (RCS-Dem 2.0), COUNTRIES ONLY.SAV")
with fs.open(FILE_RCS, "rb") as f:
    rcs_raw = pd.read_spss(f)

# West Germany (1945-1989) has no ISO3 code in RCS-Dem: we give it the code of Germany
rcs_raw.loc[rcs_raw["ABBREV"] == "GFR", "ISO3"] = "DEU"

religions = pd.DataFrame({
    "iso3": rcs_raw["ISO3"].astype(str).str.strip(),
    "year": rcs_raw["YEAR"],
    "population": rcs_raw["TOTPP"],
    "christians": rcs_raw["CHRPC"],
    "muslims": rcs_raw["MUSPC"],
    "jews": rcs_raw["JEWPC"],
    "buddhists": rcs_raw["BUDPC"],
    "hindus": rcs_raw["HINPC"],
    "no_religion": rcs_raw["NREPC"],
    "catholics": rcs_raw["CATPC"],
    "protestants": rcs_raw["PRTPC"],
    "orthodox": rcs_raw["ORTPC"],
    "unknown": rcs_raw["UNKPC"],
})

# Other religions = what is left when we remove the main religions and the unknown
main_religions = ["christians", "muslims", "jews", "buddhists", "hindus", "no_religion"]
religions["other_religions"] = 100 - religions[main_religions].sum(axis=1) - religions["unknown"].fillna(0)
# Small negative values come from rounding in the data: we put them to 0
religions["other_religions"] = religions["other_religions"].clip(lower=0)
# If a country has no data on religions, the other religions are unknown too
religions.loc[religions["christians"].isna(), "other_religions"] = np.nan
# Some old states have no ISO3 code: we remove them
religions = religions[religions["iso3"] != ""]
religions = religions.dropna(subset=["year"])
religions["year"] = religions["year"].astype(int)
religions.describe()

### GDP per capita, region and income (World Bank)

GDP per capita in constant 2015 dollars (indicator `NY.GDP.PCAP.KD`), downloaded from https://data.worldbank.org/indicator/NY.GDP.PCAP.KD (Download, CSV).

In [ ]:
FILE_GDP = "avibert/diffusion/API_NY.GDP.PCAP.KD_DS2_en_csv_v2_71952.csv"
with fs.open(FILE_GDP, "rb") as f:
    gdp_raw = pd.read_csv(f, skiprows=4)   # the first 4 lines are a title, not the table

# The last column is empty (the lines end with a comma): we remove the empty columns
gdp_raw = gdp_raw.dropna(axis=1, how="all")

# One column per year -> one row per country and per year
wb = gdp_raw.drop(columns=["Country Name", "Indicator Name", "Indicator Code"])
wb = wb.melt(id_vars="Country Code", var_name="year", value_name="gdp_per_capita")
wb = wb.rename(columns={"Country Code": "iso3"})
wb["year"] = wb["year"].astype(int)
wb.head()

In [ ]:
# Region and income group of each country (current classification)
FILE_COUNTRIES = "avibert/diffusion/Metadata_Country_API_NY.GDP.PCAP.KD_DS2_en_csv_v2_71952.csv"
with fs.open(FILE_COUNTRIES, "rb") as f:
    wb_countries = pd.read_csv(f)

wb_countries = wb_countries[["Country Code", "Region", "IncomeGroup"]]
wb_countries = wb_countries.rename(columns={"Country Code": "iso3", "Region": "region", "IncomeGroup": "income"})
# The aggregates (World, Euro area...) have no region: we remove them
wb_countries = wb_countries.dropna(subset=["region"])
wb_countries.head()

### Join the trips with the data of the closest year

RCS-Dem stops in 2015. For each visit, we take the closest observation (`merge_asof`), with at most 10 years of difference, and we keep the gap in years to check it. For the visits after 2015 (Francis), we therefore use the 2015 data.

In [ ]:
religions = religions.rename(columns={"year": "year_religion"}).sort_values("year_religion")

visits = pd.merge_asof(
    trips.sort_values("year"),
    religions,
    left_on="year", right_on="year_religion", by="iso3",
    direction="nearest", tolerance=10,
)
visits["gap_religion"] = visits["year_religion"] - visits["year"]

In [ ]:
# GDP: closest year, at most 5 years of difference
wb_sorted = wb.rename(columns={"year": "year_wb"}).sort_values("year_wb")
visits = pd.merge_asof(visits.sort_values("year"), wb_sorted,
                       left_on="year", right_on="year_wb", by="iso3",
                       direction="nearest", tolerance=5)
visits = visits.merge(wb_countries, on="iso3", how="left")

# Majority religion: we only compare the main religions and the other religions
majority_columns = main_religions + ["other_religions"]
# (dropna: we only compute the majority for the countries that have data)
visits["majority"] = visits[majority_columns].dropna(how="all").idxmax(axis=1)

### Check the join

In [ ]:
# For each pope: how many visits without data, and what is the average gap in years?
visits.groupby("pope").agg(
    nb_visits=("iso3", "size"),
    no_religion_data=("christians", lambda x: x.isna().sum()),
    no_catholic_data=("catholics", lambda x: x.isna().sum()),
    mean_gap_religion=("gap_religion", lambda x: x.abs().mean()),
).round(1)

In [ ]:
# Visited countries for which we found no religious data
visits.loc[visits["christians"].isna(), ["pope", "year", "country", "iso3"]]

In [ ]:
# Visits after 2015: the religious data is older than the trip
visits.loc[visits["year"] > 2015].groupby("pope")["gap_religion"].describe()

In [ ]:
# Visited countries where the other religions are the majority
visits.loc[visits["majority"] == "other_religions", ["pope", "year", "country", "christians", "other_religions"]].round(1)

## 2. Overview of the trips

In [ ]:
# One trip = one trip title for a pope and a year
trips["trip_id"] = trips["pope"] + " " + trips["year"].astype(str) + " " + trips["trip"]

overview = trips.groupby("pope").agg(
    nb_trips=("trip_id", "nunique"),
    nb_visits=("iso3", "size"),
    nb_countries=("iso3", "nunique"),
).loc[POPES]
overview["pontificate_length"] = [end - start for start, end in PONTIFICATES.values()]
overview["trips_per_year"] = (overview["nb_trips"] / overview["pontificate_length"]).round(1)
overview

In [ ]:
per_year = trips.groupby(["year", "pope"])["trip_id"].nunique().unstack(fill_value=0)[POPES]
per_year.plot.bar(stacked=True, figsize=(12, 4), width=0.8)
plt.ylabel("Number of trips")
plt.title("Trips outside Italy each year")
plt.show()

In [ ]:
# The most visited countries
trips.groupby("country")["trip_id"].nunique().sort_values(ascending=False).head(15)

### Maps of the visited countries

In [ ]:
world = gpd.read_file("https://naciscdn.org/naturalearth/50m/cultural/ne_50m_admin_0_countries.zip")
world = world[["NAME", "ISO_A3_EH", "geometry"]].rename(columns={"NAME": "name", "ISO_A3_EH": "iso3"})
world = world[world["name"] != "Antarctica"]
world.head()

In [ ]:
# Map 1: number of popes (among the three) who visited each country
nb_popes = trips.groupby("iso3")["pope"].nunique().reset_index(name="nb_popes")
world_popes = world.merge(nb_popes, on="iso3", how="left")
world_popes["nb_popes"] = world_popes["nb_popes"].fillna(0)

colors = {1: "#9ecae1", 2: "#4292c6", 3: "#08306b"}

fig, ax = plt.subplots(figsize=(14, 7))
world_popes.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.3)
for n, color in colors.items():
    world_popes[world_popes["nb_popes"] == n].plot(ax=ax, color=color, edgecolor="white", linewidth=0.3)

legend = [Patch(color="lightgrey", label="not visited")]
for n, color in colors.items():
    legend.append(Patch(color=color, label=f"visited by {n} pope(s)"))
ax.legend(handles=legend, loc="lower left")
ax.set_title("Countries visited by John Paul II, Benedict XVI and Francis")
ax.set_axis_off()
plt.show()

In [ ]:
# Maps 2 to 4: the countries visited by each pope
fig, axes = plt.subplots(3, 1, figsize=(12, 18))
for ax, pope in zip(axes, POPES):
    visited_iso3 = trips.loc[trips["pope"] == pope, "iso3"]
    world.plot(ax=ax, color="lightgrey", edgecolor="white", linewidth=0.3)
    world[world["iso3"].isin(visited_iso3)].plot(ax=ax, color="#2171b5", edgecolor="white", linewidth=0.3)
    ax.set_title(f"Countries visited by {pope}")
    ax.set_axis_off()
plt.show()

> To write: pace of the trips, most visited countries, missing regions.

## 3. Who do the popes speak to?

To compare the visited countries with the others, we build a table with **one row per pope and per country of the world**, and a column `visited` which is True if the pope went there. Each country is counted once, even if the pope went there several times.

We use the religious data of the middle year of the pontificate (2015 for Francis, because RCS-Dem stops in 2015).

In [ ]:
rows = []
for pope, (start, end) in PONTIFICATES.items():
    middle_year = min((start + end) // 2, 2015)
    table = religions[religions["year_religion"] == middle_year].copy()
    table["pope"] = pope
    visited_iso3 = trips.loc[trips["pope"] == pope, "iso3"]
    table["visited"] = table["iso3"].isin(visited_iso3)
    rows.append(table)

countries = pd.concat(rows, ignore_index=True)
countries["majority"] = countries[majority_columns].dropna(how="all").idxmax(axis=1)

# Number of visited countries and number of countries for each pope
countries.groupby("pope")["visited"].agg(["sum", "size"]).loc[POPES]

### Share of Catholics in the visited countries

In [ ]:
visited_countries = countries[countries["visited"]]
visited_countries.groupby("pope")["catholics"].describe().loc[POPES].round(1)

In [ ]:
# Same thing for the countries that were not visited, to compare
countries[~countries["visited"]].groupby("pope")["catholics"].describe().loc[POPES].round(1)

### How many visited countries are Christian or Catholic?

- Christian: Christians are the largest religion of the country (`majority` column).
- Catholic: Catholics are more than 50 % of the population.

In [ ]:
visited_countries = countries[countries["visited"]].copy()
visited_countries["christian_majority"] = visited_countries["majority"] == "christians"
visited_countries["catholic_majority"] = visited_countries["catholics"] > 50

result = visited_countries.groupby("pope")[["christian_majority", "catholic_majority"]].sum()
result["nb_visited_countries"] = visited_countries.groupby("pope").size()
result["% christian_majority"] = (result["christian_majority"] / result["nb_visited_countries"] * 100).round(1)
result["% catholic_majority"] = (result["catholic_majority"] / result["nb_visited_countries"] * 100).round(1)
result.loc[POPES]

### How many visited countries are more Catholic than the others?

For each pope, we compute the average share of Catholics in the countries he did **not** visit, and we count the visited countries above this average.

In [ ]:
results = []
for pope in POPES:
    table = countries[(countries["pope"] == pope) & countries["catholics"].notna()]
    visited = table[table["visited"]]
    not_visited = table[~table["visited"]]

    mean_not_visited = not_visited["catholics"].mean()
    with_population = table[table["population"].notna()]
    world_share = np.average(with_population["catholics"], weights=with_population["population"])

    results.append({
        "pope": pope,
        "mean visited": visited["catholics"].mean(),
        "mean not visited": mean_not_visited,
        "world share (weighted)": world_share,
        "nb visited above mean not visited": (visited["catholics"] > mean_not_visited).sum(),
        "% visited above mean not visited": (visited["catholics"] > mean_not_visited).mean() * 100,
    })

pd.DataFrame(results).set_index("pope").round(1)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, pope in zip(axes, POPES):
    table = countries[countries["pope"] == pope]
    ax.hist(table.loc[~table["visited"], "catholics"].dropna(), bins=20, alpha=0.5, label="not visited", density=True)
    ax.hist(table.loc[table["visited"], "catholics"].dropna(), bins=20, alpha=0.5, label="visited", density=True)
    ax.set_title(pope)
    ax.set_xlabel("Share of Catholics (%)")
axes[0].legend()
plt.show()

In [ ]:
# Differences between regions (visits, with the data of the year of the visit)
visits.groupby(["region", "pope"])["catholics"].mean().unstack()[POPES].round(1)

## 4. How Christian population evolves in visited countries ?

For each visit, we compute the change of the share of Catholics in the visited country during the **5 years before the visit** (in percentage points). We compare it with the change of the **average share of Catholics of all countries** during the same 5 years (each country counts for one, like in part 3).

RCS-Dem stops in 2015: for the visits after 2015, we use the change between 2010 and 2015.

In [ ]:
# Average share of Catholics of all countries, each year
mean_catholics = religions.groupby("year_religion")["catholics"].mean().reset_index(name="mean_catholics")

# The first visit is in 1979, so the first year we need is 1974
mean_catholics[mean_catholics["year_religion"] >= 1974].plot(x="year_religion", y="mean_catholics", legend=False)
plt.xlabel("Year")
plt.ylabel("Share of Catholics (%)")
plt.title("Average share of Catholics of all countries")
plt.show()

In [ ]:
# The 5 years before the visit (the data stops in 2015)
visits["year_end"] = visits["year"].clip(upper=2015)
visits["year_start"] = visits["year_end"] - 5

# Share of Catholics in the visited country at the start and at the end
share_start = religions[["iso3", "year_religion", "catholics"]].rename(
    columns={"year_religion": "year_start", "catholics": "catholics_start"})
share_end = religions[["iso3", "year_religion", "catholics"]].rename(
    columns={"year_religion": "year_end", "catholics": "catholics_end"})
visits = visits.merge(share_start, on=["iso3", "year_start"], how="left")
visits = visits.merge(share_end, on=["iso3", "year_end"], how="left")

# Average share of Catholics of all countries at the start and at the end
mean_start = mean_catholics.rename(columns={"year_religion": "year_start", "mean_catholics": "mean_start"})
mean_end = mean_catholics.rename(columns={"year_religion": "year_end", "mean_catholics": "mean_end"})
visits = visits.merge(mean_start, on="year_start", how="left")
visits = visits.merge(mean_end, on="year_end", how="left")

# Changes in percentage points
visits["change_country"] = visits["catholics_end"] - visits["catholics_start"]
visits["change_mean"] = visits["mean_end"] - visits["mean_start"]
visits["change_vs_mean"] = visits["change_country"] - visits["change_mean"]

# Check: how many visits without a change?
visits.groupby("pope")["change_country"].apply(lambda x: x.isna().sum())

### Descriptive statistics

In [ ]:
changes = visits.dropna(subset=["change_country"]).copy()
changes.groupby("pope")["change_country"].describe().loc[POPES].round(2)

In [ ]:
# Average change in the visited countries and in all countries
changes.groupby("pope")[["change_country", "change_mean", "change_vs_mean"]].mean().loc[POPES].round(2)

In [ ]:
# Share of visits where Catholicism was growing, and growing faster than on average
changes["growing"] = changes["change_country"] > 0
changes["faster_than_mean"] = changes["change_vs_mean"] > 0

(changes.groupby("pope")[["growing", "faster_than_mean"]].mean() * 100).loc[POPES].round(1)

In [ ]:
changes.boxplot(column="change_vs_mean", by="pope")
plt.axhline(0, color="grey")
plt.title("Change of the share of Catholics, compared with the average of all countries (5 years before the visit)")
plt.suptitle("")
plt.xlabel("")
plt.ylabel("Percentage points")
plt.show()